### Run the servers

Make sure you are on the VPN and the AutoScript server is running. Then start the asyncroscopy Tango servers from the repository root:

```bash
uv run scripts/run_servers.py
```


### Imports


In [ ]:
import os
import json
import time
import h5py
import xml.etree.ElementTree as ET

import tango
import numpy as np
import matplotlib.pyplot as plt
from tiled.client import from_uri

from autoscript_tem_microscope_client import TemMicroscopeClient
from autoscript_tem_microscope_client.enumerations import EdsDetectorType
from autoscript_tem_microscope_client.enumerations import CameraType, RegionCoordinateSystem, ExposureTimeType
from autoscript_tem_microscope_client.structures import Region, Rectangle
from autoscript_tem_microscope_client.structures import StemAcquisitionSettings, EdsAcquisitionSettings, RunOptiStemSettings, CameraAcquisitionSettings, StemDataSettings

%matplotlib ipympl


In [1]:
from autoscript_tem_microscope_client import TemMicroscopeClient

mic = TemMicroscopeClient()

gatan_ip = '10.10.11.52'
gatan_port = 9095

mic.connect(gatan_ip, gatan_port)

Client connecting to [10.10.11.52:9095]...
Client connected to [10.10.11.52:9095]


In [7]:
mic.service.system.name

'Talos'

In [3]:
mic.vacuum.state

'Ready'

In [ ]:
mic.vacuum.s

In [ ]:
mic = TemMicroscopeClient()

gatan_ip = '10.46.217.242'
gatan_port = 9094

mic.connect(gatan_ip, gatan_port)

In [ ]:
detector_type = 'HAADF'
dwell_time = 1e-6
imsize = 2048
im = mic.acquisition.acquire_stem_image(detector_type, imsize, dwell_time)

In [ ]:
detector_list = ['HAADF', 'BF']
scan_region = (0,0,.5,.5)
detector_list = [d.upper() for d in detector_list]
settings = StemAcquisitionSettings(dwell_time=dwell_time, detector_types=detector_list, size=imsize, region=Region(RegionCoordinateSystem.RELATIVE, Rectangle(*scan_region)))
adorned_list = mic.acquisition.acquire_stem_images_advanced(settings)
adorned_list


In [ ]:
path = '/Users/austin/Desktop/testing/'

# autoscript data
image = im.data
metadata_xml = im.metadata.metadata_as_xml

# ------------------------------------------------
# TIFF timing
# ------------------------------------------------
t0 = time.perf_counter()
im.save(path + 'im0.tiff')
print(f'TIFF save time: {time.perf_counter() - t0:.4f} s')

# ------------------------------------------------
# HDF5 timing (raw XML blob)
# ------------------------------------------------
t0 = time.perf_counter()

with h5py.File(path + 'im0_rawxml.h5', 'w') as f:
    f.create_dataset('image', data=image, compression=None)
    f.create_dataset('metadata_xml', data=metadata_xml.encode('utf-8'))

print(f'HDF5 raw XML save time: {time.perf_counter() - t0:.4f} s')

# ------------------------------------------------
# HDF5 timing (parsed XML attributes)
# ------------------------------------------------
t0 = time.perf_counter()

root = ET.fromstring(metadata_xml)

with h5py.File(path + 'im0_attrs.h5', 'w') as f:

    dset = f.create_dataset('image', data=image, compression=None)

    for elem in root.iter():

        # leaf nodes only
        if elem.text and elem.text.strip():

            key = elem.tag
            value = elem.text.strip()

            # avoid duplicate names
            if key in dset.attrs:
                key = f'{key}_{len(dset.attrs)}'

            dset.attrs[key] = value

print(f'HDF5 parsed attrs save time: {time.perf_counter() - t0:.4f} s')

In [ ]:
im = mic.acquisition.acquire_stem_image(detector_type, imsize, dwell_time)

image = im.data
metadata_xml = im.metadata.metadata_as_xml

root = ET.fromstring(metadata_xml)

with h5py.File(path + 'im0_attrs.h5', 'w') as f:
    dset = f.create_dataset('image', data=image, compression=None)
    for elem in root.iter():
        # leaf nodes only
        if elem.text and elem.text.strip():
            key = elem.tag
            value = elem.text.strip()
            if key in dset.attrs:
                key = f'{key}_{len(dset.attrs)}'
            dset.attrs[key] = value

### Connect to devices


In [ ]:
DB_HOST = "127.0.0.1"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

server_names = ['stage', 'scan', 'eds', 'camera', 'data', 'microscope']

for name in server_names:
    device_name = f"asyncroscopy/{name}/default"
    proxy = tango.DeviceProxy(device_name)
    proxy.set_timeout_millis(120_000)
    proxy.ping()
    print(device_name, proxy.state())


scan = tango.DeviceProxy("asyncroscopy/scan/default")
microscope = tango.DeviceProxy("asyncroscopy/microscope/default")
data = tango.DeviceProxy("asyncroscopy/data/default")


### Start Tiled data server


In [ ]:
TILED_HOST = '127.0.0.1'
TILED_PORT = 9091
save_path = '/Users/austin/Desktop/testing'

data.host = TILED_HOST
data.port = TILED_PORT
data.save_path = save_path

if str(data.tiled_server).lower() != "yes":
    print("Tiled server is not responding; starting it from the DATA device...")
    config = json.loads(data.start_tiled_server())
else:
    print("Tiled server is already running.")
    config = json.loads(data.get_config())

print(json.dumps(config, indent=2))

client = from_uri(config.get("uri", f"http://{TILED_HOST}:{TILED_PORT}"))
print("Tiled keys:", list(client))


### Configure scan


In [ ]:
scan.dwell_time = 1e-6
scan.imsize = 512
scan.scan_region = [0, 0, 1, 1]

print("dwell_time :", scan.dwell_time)
print("image size :", scan.imsize)
print("scan region:", list(scan.scan_region))


### Acquire a HAADF image


In [ ]:
key = microscope.acquire_scanned_image()
node = client[key]
image = np.asarray(node.read())
metadata = dict(node.metadata)

print("Tiled key  :", key)
print("Metadata   :", metadata)
print("Image shape:", image.shape)
print("Image dtype:", image.dtype)


### Display the image


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap="gray", interpolation="none")
ax.set_title(f"HAADF - dwell {scan.dwell_time * 1e6:.1f} us")
ax.axis("off")
plt.tight_layout()


### Acquire multiple detectors


In [ ]:
scan.dwell_time = 1e-6
scan.imsize = 512
scan.scan_region = [0, 0, 1, 1]

# think we can delete this from the
scan.haadf = True
scan.bf = True

keys = json.loads(microscope.acquire_images(detector_list = ["HAADF", "BF"]))
images = []

for key in keys:
    node = client[key]
    img = np.asarray(node.read())
    metadata = dict(node.metadata)
    detector_name = metadata.get("detector", key)
    images.append((detector_name, key, img, metadata))
    print(detector_name, key, img.shape, img.dtype)


In [ ]:
fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 5))
if len(images) == 1:
    axes = [axes]

for ax, (detector_name, key, img, img_metadata) in zip(axes, images):
    im = ax.imshow(img, cmap="gray")
    ax.set_title(str(detector_name).upper())
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()


### Acquire a cropped rectangle scan


In [ ]:
scan.Activate(["haadf"])
scan.dwell_time = 1e-6
scan.imsize = 512
scan.scan_region = [0, 0.4, 1, 0.2]

key = microscope.acquire_scanned_image_advanced()
node = client[key]
cropped_image = np.asarray(node.read())
cropped_metadata = dict(node.metadata)

print("Tiled key  :", key)
print("Metadata   :", cropped_metadata)
print("Image shape:", cropped_image.shape)
print("Image dtype:", cropped_image.dtype)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cropped_image, cmap="gray")
ax.set_title(f"Cropped scan: {key}")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
